# OrganicAI — 협력 층위(L0~L4) 판정 파이프라인

시나리오 하나를 넣으면 두 에이전트의 협력 층위를 판정한다.

**처음 쓴다면 `docs/SCENARIO_TEST.md`를 먼저 읽을 것.** (API 키 발급 · Colab 설정 · 에러 대처)

모델 배정 근거는 `docs/MODEL_ASSIGNMENT.md`, 데이터 저장 규칙은 `README.md` 에 있다.

## 실행 순서 요약

| 단계 | 언제 |
|---|---|
| 0. 설치 | 세션 최초 1회 |
| 1~3 | 매 세션 필수 (순서 지킬 것) |
| 4~5 | 시나리오를 새로 썼을 때 |
| 6 | 데이터 수집 |

> ⚠️ **런타임을 재시작했다면 0번을 건너뛰고 1번부터 다시 실행한다.**
> `!git pull` 로 코드를 받은 뒤에도 **반드시 런타임을 재시작**해야 새 코드가 반영된다.

## 0. 설치 (세션 최초 1회)

In [ ]:
!git clone -b yr https://github.com/ewha-oi/OrganicAI.git
%cd OrganicAI
!pip install -r requirements.txt -q

import sys
sys.path.append('src')
print("Python:", sys.version)

## 1. 세션 시작

**런타임 재시작 후에는 여기서부터 실행한다.**

In [ ]:
%cd /content/OrganicAI
import sys
sys.path.append('src')
!git log --oneline -1

In [ ]:
from google.colab import userdata


def _secret(name):
    # 등록돼 있지 않으면 userdata.get 이 예외를 던진다. gemini 는 지금 미사용이므로
    # 없다고 세션을 멈출 이유가 없다 -> None 으로 넘기고 아래에서 표시만 한다.
    try:
        return userdata.get(name)
    except Exception:
        return None


API_KEYS = {
    "gemini": _secret('GEMINI_API_KEY'),   # 현재 미사용 - 없어도 정상 (3절 shim 이 대체)
    "groq":   _secret('GROQ_API_KEY'),     # 필수
}

for k, v in API_KEYS.items():
    mark = 'OK' if v else ('없음 (미사용이라 무방)' if k == 'gemini'
                           else '!! 없음 - Secrets 이름/노트북 액세스 토글 확인')
    print(f"{k:8s} {mark}")

In [ ]:
# 모델 배정. coop_pipeline 을 임포트하는 어떤 셀보다 먼저 실행한다
# (llm.py 가 임포트 시점에 한 번만 읽는다. 값을 바꿨으면 런타임 재시작).
# 배정 근거와 재배정 절차: docs/MODEL_ASSIGNMENT.md

import os
os.environ["COOP_JUDGE_PROVIDER"] = "groq"                 # anthropic / groq / gemini
os.environ["COOP_JUDGE_MODEL"]    = "qwen/qwen3.6-27b"
os.environ["COOP_ALPHA_MODEL"]    = "openai/gpt-oss-120b"
os.environ["COOP_BETA_MODEL"]     = "openai/gpt-oss-20b"   # 기본값 llama-3.3-70b 는 이 계정에 없다

for k in ("COOP_JUDGE_PROVIDER", "COOP_JUDGE_MODEL",
          "COOP_ALPHA_MODEL", "COOP_BETA_MODEL"):
    print(f"{k:22s} {os.environ[k]}")


## 2. 환경 점검 (API 호출 없음, 무료)

여기서 걸리는 문제는 실행해도 똑같이 걸린다. 먼저 통과시킬 것.

In [ ]:
!python -m pytest tests/ -q
!python tools/dryrun_frame.py

In [ ]:
# 시나리오 형식 + 현재 모델 구성 확인
from coop_pipeline.runner import check_scenario_dir
from coop_pipeline.llm import MODELS, judge_provider, judge_model, judge_key_name

check_scenario_dir("scenarios")

print(f"\nalpha : {MODELS['alpha']}")
print(f"beta  : {MODELS['beta']}")
print(f"judge : {judge_provider()}:{judge_model()}   (필요한 키: {judge_key_name()})")

## 3. [임시] alpha 모델 대체

Gemini 접근이 막혀서(전 모델 403/404) alpha 를 Groq 모델로 돌린다.
아래 셀은 추론 옵션과 `max_tokens` 를 붙이고 `<think>` 를 지우는 어댑터다.
배경은 `docs/MODEL_ASSIGNMENT.md`.

**Gemini 가 복구되면 이 셀만 건너뛰면 원래 설계로 돌아간다** (코드 수정 불필요).

In [ ]:
import re, types
from groq import Groq
from coop_pipeline import agents, llm
from coop_pipeline.runner import load_scenario

GROQ_KEY = API_KEYS["groq"]

# 모델 ID 출처는 1절 배정 셀 하나뿐이다. 여기에 하드코딩 금지.
ALPHA_MODEL = llm.MODELS["alpha"]
_THINK = re.compile(r"<think>.*?</think>\s*", re.S)

# 이 모델이 받는 추론 옵션을 런타임에 찾는다 (모델마다 다르고, 틀리면 400).
EXTRA = {}
for cand in ({"reasoning_effort": "low",  "reasoning_format": "hidden"},
             {"reasoning_effort": "none", "reasoning_format": "hidden"},
             {"reasoning_effort": "low"},
             {"reasoning_effort": "none"},
             {"reasoning_format": "hidden"},
             {}):
    try:
        Groq(api_key=GROQ_KEY).chat.completions.create(
            model=ALPHA_MODEL, messages=[{"role": "user", "content": "ping"}],
            max_tokens=64, **cand)
        EXTRA = cand
        break
    except Exception as e:
        print("불가:", cand, "|", str(e)[:90])
print(f"alpha = {ALPHA_MODEL}")
print("사용할 옵션:", EXTRA, "\n")


class _Resp:
    def __init__(self, text): self.text = text


class _GroqModel:
    def __init__(self, model_id): self.model_id = model_id

    def generate_content(self, prompt):
        r = Groq(api_key=GROQ_KEY).chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=agents.TEMPERATURE,
            max_tokens=4096,          # 없으면 추론 토큰에 다 쓰고 빈 응답이 온다
            **EXTRA,
        )
        text = _THINK.sub("", r.choices[0].message.content or "").strip()
        if not text:
            raise RuntimeError(
                f"alpha({self.model_id}) 빈 응답 — max_tokens 부족이거나 "
                f"추론이 출력 한도를 소진함. 현재 EXTRA={EXTRA}")
        return _Resp(text)


_shim = types.SimpleNamespace(configure=lambda **kw: None, GenerativeModel=_GroqModel)
agents._gemini_model = lambda: _shim
agents.RATE_LIMIT_SLEEP = 3.0   # 429 가 뜨면 6.0~10.0 으로 올린다

# 스모크 테스트는 실제 길이의 프롬프트로 한다 (짧은 ping 은 통과해도 실제에서 걸린다).
sc = load_scenario("scenarios/A4/A4_simple_energy_save.json")
v = sc["task_variants"]
task = v.get("alpha") or v["shared"]      # 비대칭/대칭 시나리오 양쪽 모두 대응
long_prompt = agents.SYSTEM_PROMPT_TEMPLATES["명시"].format(
    name="alpha", partner="beta", task=task
) + "\n\n지금까지의 대화:\n(없음)\n\n너의 다음 발언:"

out = _GroqModel(ALPHA_MODEL).generate_content(long_prompt).text
print(f"길이 {len(out)}자")
print(out[:300])
print("\nMODELS:", llm.MODELS)

## 4. 발화 태깅 점검

채점자가 발화 코드(phatic/meta/lead/arch/agree/comp)를 제대로 붙이는지 5개 사례로 본다.
**4/5 이상이면 통과.** 3/5 이하면 채점자 모델을 의심할 것.

In [ ]:
from coop_pipeline.llm import make_judge
from coop_pipeline.tagging import tag_turn

CASES = [
    ("수요일 B실로 하자.",     "좋아, 그렇게 하자.",                                 "phatic"),
    ("수요일에 하는 게 어때?",  "맞네, 나는 A실을 생각했는데 수요일이면 B실이 맞겠다.",  "agree"),
    ("수요일 B실로 하자.",     "좋아. 그런데 예산 확인도 필요해 보여.",                "comp"),
    ("예산은 200이야.",       "응, 200이지.",                                     "phatic"),
    ("회의 준비 시작하자.",    "지금 정할 건 요일이야. 시간은 나중에.",                "lead"),
]

j = make_judge(API_KEYS["groq"])
hit = 0
for prev, cur, want in CASES:
    got = tag_turn(j, [{"turn": 1, "speaker": "alpha", "text": prev}],
                   {"turn": 2, "speaker": "beta", "text": cur})
    ok = (got["codes"] == ["phatic"]) if want == "phatic" else (want in got["codes"])
    hit += ok
    print(f"{'O' if ok else 'X'} 기대={want:6s} 실제={got['codes']} ref={got['ref']}")
    print(f"   근거: {got['evidence']}")
print(f"\n{hit}/5")

## 5. 시나리오 실기동

**새 시나리오를 썼다면 여기서 한 번 돌려본다.**

통과 기준은 하나뿐이다 — **에러 없이 `L0`~`L4` 중 하나가 나오는 것.**
어느 층위가 나오든, `[FAIL] Q3` 이 뜨든 상관없다. 그건 실험 결과이지 형식 오류가 아니다.

In [ ]:
# 새 시나리오 형식 점검용 시험 실행. 수집 데이터로는 쓰지 않는다.
from coop_pipeline.runner import run_scenario

result = run_scenario(
    "scenarios/A4/A4_simple_energy_save.json",
    condition="명시",     # 또는 "묵시"
    api_keys=API_KEYS,
    n_solo=2,            # 시험용. 정식 실행은 5
    max_turns=6,         # 시험용. 정식 실행은 10
    out_dir="runs_pilot",
)

In [ ]:
# 무엇이 실제로 일어났는지 들여다보기 (에러 원인 추적용)
log = result["log"]

print("=== 1) 대화가 실제로 오갔는가 ===")
for t in log["turns"]:
    print(f"[{t['turn']}] {t['speaker']}: {t['text'][:150]}")

print("\n=== 2) 태깅이 붙었는가 ===")
for t in log["turns"]:
    print(t["turn"], t["speaker"], t["codes"], "ref:", t["ref"])

print("\n=== 3) 무엇을 채점했는가 ===")
print(log["group_output_text"][:500])
print("\n단독:", log.get("solo_grades") or log.get("solo_scores"),
      "/ 그룹:", log.get("group_grade") or log.get("group_score"))

In [ ]:
# A1 경로 점검 — 체크리스트 채점 + 퍼센타일 판정. A2/A4 와 코드 경로가 다르다.
from coop_pipeline.runner import run_scenario

res_A1 = run_scenario(
    "scenarios/A1/A1_simple_meeting_room.json",
    condition="명시",
    api_keys=API_KEYS,
    n_solo=2,
    max_turns=6,
    out_dir="runs_pilot",
)

print("\n판정 :", res_A1["level"])
print("병목 :", res_A1["stopped_at"] or "없음 (L4까지 통과)")

In [ ]:
# 저장된 로그를 다시 판정한다. API 호출 0, 무료.
!ls -R runs_pilot

from coop_pipeline.runner import classify_saved_dir
_ = classify_saved_dir("runs_pilot")

## 6. 파일럿 데이터 수집

명시/묵시 두 조건을 돌린다. 단독 산출물은 rep 마다 한 번만 만들어 두 조건이 공유한다.

셀 하나가 `(시나리오 × rep)` 목록을 만들어 순서대로 돈다.

- **이어서 돌리기** — 이미 두 조건이 다 저장된 것은 건너뛴다. 끊겼으면 셀을 그대로 다시 실행.
- **한 건이 실패해도 멈추지 않는다** — 실패 목록을 끝에 찍고, 다시 실행하면 그것만 재시도한다.
- **`BATCH`** 로 하루치씩 끊는다. Groq 무료 티어 일일 한도(특히 judge)에 걸리기 때문.

> ⚠️ `MY_REPS` 와 `SCENARIOS` 는 사람마다 다르다. rep 이 겹치면 파일이 조용히 덮어써진다
> (`README.md` 「데이터 저장 규칙」).
> 수집 파라미터(`MAX_TURNS`, `N_SOLO`, 모델)는 팀 노션의 **동결표**에 고정돼 있다 — 수집 도중에 바꾸지 않는다.

In [ ]:
# 파일럿 데이터 수집. Drive 에 직접 쓴다 (세션이 끊겨도 그때까지 모은 것은 남는다).
# 중단됐으면 이 셀을 그대로 다시 실행한다 — 이미 만든 (시나리오, rep) 은 건너뛴다.
#
# rep 분배: 김유리 1~5 / 이예영 6~10 / 김유민 11~15  — README 「데이터 저장 규칙」
# 파일명에 사람 구분이 없다. rep 이 겹치면 남의 파일을 조용히 덮어쓴다.

MY_REPS = range(1, 6)          # <-- 본인 rep 범위로 바꾼다

# 본인 담당 시나리오. 사람마다 다르므로 돌리기 전에 확인할 것.
SCENARIOS = [
    "scenarios/A1/A1_complex_gas_alarm.json",
    "scenarios/A1/A1_complex_power_outage.json",
    "scenarios/A1/A1_complex_wifi_outage.json",
    "scenarios/A1/A1_simple_meeting_room.json",
    "scenarios/A1/A1_simple_seminar_hall.json",
    "scenarios/A2/A2_complex_course_slots.json",
    "scenarios/A2/A2_complex_grading_policy.json",
    "scenarios/A2/A2_complex_ta_selection.json",
    "scenarios/A2/A2_simple_mt_venue.json",
    "scenarios/A2/A2_simple_transport.json",
    "scenarios/A4/A4_complex_career_bootcamp.json",
    "scenarios/A4/A4_complex_onboarding.json",
    "scenarios/A4/A4_complex_research_ethics.json",
    "scenarios/A4/A4_simple_festival.json",
    "scenarios/A4/A4_simple_punctuality.json",
    "scenarios/A4/A4_simple_water_save.json",
]

# 동결 파라미터 — 수집 도중 변경 금지 (노션 동결표).
MAX_TURNS = 10
N_SOLO    = 5                  # 에이전트당

# 한 번에 돌릴 (시나리오, rep) 수. Groq 무료 티어 일일 한도에 걸리므로
# 하루치씩 끊어 돌린다. 16 = rep 1회분. None 이면 남은 것을 전부.
BATCH = 16

import time
from pathlib import Path
from google.colab import drive
from coop_pipeline.runner import load_scenario, run_scenario_both_conditions

drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/OrganicAI_runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def is_done(sid, rep):
    """명시·묵시 두 조건이 다 저장돼 있어야 완료로 본다."""
    return all((OUT_DIR / f"{sid}_{c}_{rep}.json").exists() for c in ("명시", "묵시"))


# rep 을 바깥에 둔다 -> 16개를 한 바퀴 끝내고 다음 rep 으로 간다.
# 중간에 끊겨도 시나리오별 rep 수가 고르게 유지된다.
jobs = [(p, rep, load_scenario(p)["scenario_id"]) for rep in MY_REPS for p in SCENARIOS]
todo = [j for j in jobs if not is_done(j[2], j[1])]
batch = todo[:BATCH] if BATCH else todo

print(f"저장 위치 : {OUT_DIR}")
print(f"전체 {len(jobs)}건 / 완료 {len(jobs) - len(todo)}건 / 남은 것 {len(todo)}건")
print(f"이번에 돌릴 것 : {len(batch)}건")
print("=" * 70)

t_start, failed = time.time(), []
for i, (path, rep, sid) in enumerate(batch, 1):
    try:
        run_scenario_both_conditions(
            path, api_keys=API_KEYS, replicate=rep,
            n_solo=N_SOLO, max_turns=MAX_TURNS, out_dir=str(OUT_DIR), verbose=False)
        print(f"[{i}/{len(batch)}] OK   rep={rep}  {sid}"
              f"  (누적 {(time.time() - t_start) / 60:.0f}분)")
    except Exception as e:
        # 한 건이 실패해도 나머지를 계속 돈다. 실패분은 다시 실행하면 재시도된다.
        failed.append((sid, rep, str(e)[:150]))
        print(f"[{i}/{len(batch)}] FAIL rep={rep}  {sid}: {str(e)[:150]}")

n_files = len(list(OUT_DIR.glob("*.json")))
print(f"\n종료 — {(time.time() - t_start) / 60:.0f}분, 실패 {len(failed)}건, "
      f"Drive 에 완성 로그 {n_files}개")
for sid, rep, msg in failed:
    print(f"  {sid} rep={rep}: {msg}")
if len(todo) > len(batch):
    print(f"\n아직 {len(todo) - len(batch)}건 남았다. 한도가 회복되면 이 셀을 다시 실행할 것.")

In [ ]:
# 수집 현황 + 전체 판정 (한 폴더에 모인 것을 한 번에). API 호출 0, 무료.
from collections import Counter
from pathlib import Path
from google.colab import drive
from coop_pipeline.runner import classify_saved_dir

drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/OrganicAI_runs"

# 파일명 규칙: {scenario_id}_{condition}_{rep}.json  (raw/ 하위는 원본이라 제외된다)
rows, skipped = [], []
for p in sorted(Path(OUT_DIR).glob("*.json")):
    parts = p.stem.rsplit("_", 2)          # [scenario_id, condition, rep]
    if len(parts) == 3 and parts[2].isdigit() and parts[1] in ("명시", "묵시"):
        rows.append((parts[0], parts[1], int(parts[2])))
    else:
        skipped.append(p.name)             # 규칙을 안 따르는 파일은 세지 않는다

reps = sorted({r for _, _, r in rows})
print(f"로그 {len(rows)}개 / rep {reps}")
if skipped:
    print(f"(규칙 밖 파일 {len(skipped)}개 무시: {skipped[:3]})")

# 조건·시나리오별 개수 — 한쪽만 많으면 조건 간 비교가 기운다.
for (sid, cond), n in sorted(Counter((s, c) for s, c, _ in rows).items()):
    print(f"  {sid:34s} {cond}  {n}회")

print()
_ = classify_saved_dir(OUT_DIR)


In [ ]:
# 임계값 민감도 관찰. v1 사본을 메모리에서만 바꾼다 (로그도 configs/ 도 안 건드린다).
# 결과에 맞춰 기준을 고치지 말 것 — v2 확정은 수집이 끝난 뒤. API 호출 0, 무료.
from coop_pipeline.runner import classify_saved_dir
from coop_pipeline import load_thresholds
from google.colab import drive

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/OrganicAI_runs"    # 6절과 같은 폴더

for gap in (2, 1.5, 1, 0.5):
    print(f"\n### grade_gap_min = {gap}")
    classify_saved_dir(OUT_DIR, dict(load_thresholds("v1"), grade_gap_min=gap))


## 부록

In [ ]:
# 최신 코드 받아오기. 받은 뒤에는 반드시 런타임을 재시작하고 1번부터 다시 실행할 것.
!git pull
!git log --oneline -3

In [ ]:
# Groq 에서 지금 쓸 수 있는 모델 목록. 모델이 퇴역했을 때 대체 ID 를 여기서 고른다.
# 고른 뒤 그냥 넣지 말 것 — 재배정 절차는 docs/MODEL_ASSIGNMENT.md
from groq import Groq

for m in sorted(x.id for x in Groq(api_key=API_KEYS["groq"]).models.list().data):
    print("  ", m)
